# Семинар: индексация и булев поиск

Сегодня руками соберём поисковый индекс и посмотрим, как то же самое устроено в готовом движке.

План:
1. Наивный поиск полным перебором — и почему так нельзя.
2. Свой обратный индекс на чистом Python: постинг-листы, пересечение, И/ИЛИ-запросы. **Это ровно то, что нужно сделать в ДЗ1.**
3. Тот же поиск в движке `whoosh`: схема, индексация, парсер запросов, анализатор.
4. Задания.

Перед запуском положите рядом с ноутбуком файлы `docs.tsv` и `queries.numerate.txt`.

In [1]:
# !pip install whoosh==2.7.4

## 0. Данные

Коллекция — 2781 веб-документ. Тексты уже прошли препроцессинг: приведены к леммам в верхнем регистре, пунктуация выброшена.
Как это делается, будет в теме про предобработку; сегодня считаем, что документы приходят уже в таком виде.

In [2]:
import os

DOCS_TSV = "docs.tsv"
QUERIES_TXT = "queries.numerate.txt"

for path in (DOCS_TSV, QUERIES_TXT):
    assert os.path.exists(path), f"Положите {path} рядом с ноутбуком"

print(f"{DOCS_TSV}: {os.path.getsize(DOCS_TSV) / 2**20:.0f} МБ\n")
with open(DOCS_TSV, encoding="utf-8") as f:
    for _ in range(2):
        print(next(f).rstrip()[:220], "...\n")

docs.tsv: 20 МБ

000000001	 БОЛЬШОЙ ПРОБЛЕМА С ЗАЕЗД ВЫЕЗД В ПЕРСПЕКТИВА БЫТЬ НАМНОГО ПЛОХОЙ КАК КАК ДАЛЕКО НЕ	 БОЛЬШОЙ ПРОБЛЕМА С ЗАЕЗД ВЫЕЗД В ПЕРСПЕКТИВА БЫТЬ НАМНОГО ПЛОХОЙ КАК КАК ДАЛЕКО НЕ ВЕСЬ ДОМ ЗАСЕЛИТЬ ОЧЕНЬ МНОГО СОЦИАЛЬНЫЙ Ж ...

000000002	 ГЛАВНЫЙ ПЛЮС АШАН В ТО ЧТО В ЭТОТ МАГАЗИН ПОПЫТАТЬСЯ СОБРАТЬ МАКСИМАЛЬНЫЙ КОЛИЧЕСТВО ТОВАР	 ГЛАВНЫЙ ПЛЮС АШАН В ТО ЧТО В ЭТОТ МАГАЗИН ПОПЫТАТЬСЯ СОБРАТЬ МАКСИМАЛЬНЫЙ КОЛИЧЕСТВО ТОВАР Я НРАВИТЬСЯ ПРОГУЛИВАТЬСЯ  ...



Формат `docs.tsv`: `docid`, `title`, `body`, разделённые табуляцией.

Формат `queries.numerate.txt`: `номер запроса`, таб, текст запроса. Скобки — это синонимы:
`(ВОЙНА|БОЙ|СРАЖЕНИЕ)` значит «любое из этих слов», а пробел между группами значит «и то, и другое».

In [3]:
docs = {}  # docid -> (title, body)
with open(DOCS_TSV, encoding="utf-8") as f:
    for line in f:
        docid, title, body = line.rstrip("\n").split("\t")
        docs[int(docid)] = (title.strip(), body.strip())

queries = {}  # номер запроса -> текст
with open(QUERIES_TXT, encoding="utf-8") as f:
    for line in f:
        qid, text = line.rstrip("\n").split("\t")
        queries[int(qid)] = text

print(len(docs), "документов,", len(queries), "запросов\n")
for qid in (11, 13, 15):
    print(qid, queries[qid])

2781 документов, 325 запросов

11 (1|I) 89161234567 (НАЙТИ|ПОИСК|ИСКАТЬ|ВЫЯСНИТЬ) КОМУ (ПРИНАДЛЕЖАТЬ|ИМЕТЬ) (ЭТОТ|ETOT) (НОМЕР|КОМНАТА|НОМЕРОК)
13 (1|I) (ВОЙНА|БОЙ|СРАЖЕНИЕ|БИТВА) (ГЕРМАНИЯ|ФРГ|GERMANY) (СОЮЗ|СССР|ОБЪЕДИНЕНИЕ)
15 (1|I) (ВРАЧ|ДОКТОР|МЕДИК) (БОЛЬНИЦА|КЛИНИКА|СТАЦИОНАР) (ЛЕЧЕНИЕ|ТЕРАПИЯ)


## 1. Наивный поиск: читаем всю коллекцию на каждый запрос

Это то, что делает `grep`. Разберём запрос на группы и для каждого документа проверим, что нашлось хотя бы одно слово из каждой группы.

In [4]:
import re
import time

WORD_RE = re.compile(r"\w+")


def preprocess(text):
    r"""Токенизация как в шаблоне ДЗ1: nltk.tokenize.RegexpTokenizer(r'\w+') + приведение к нижнему регистру."""
    return [token.lower() for token in WORD_RE.findall(text)]


def parse_query(query_text):
    """'(A|B) C' -> [['a', 'b'], ['c']]: список И-групп, внутри группы -- ИЛИ."""
    groups = []
    for chunk in re.findall(r"\([^)]*\)|\S+", query_text):
        variants = preprocess(chunk)
        if variants:
            groups.append(variants)
    return groups


print(queries[100])
print(parse_query(queries[100]))

КАСАТЬСЯ КЛИЕНТ ПОЙТИ
[['касаться'], ['клиент'], ['пойти']]


In [5]:
def naive_search(query_text):
    groups = parse_query(query_text)
    found = []
    for docid, (title, body) in docs.items():
        tokens = set(preprocess(title)) | set(preprocess(body))
        if all(
            any(
                variant in tokens 
                for variant in group
                ) 
                for group in groups
            ):
            found.append(docid)
    return found


start = time.time()
for qid in (13, 14, 15):
    hits = naive_search(queries[qid])
    print(f"запрос {qid}: найдено {len(hits)}, первые docid: {hits[:5]}")
naive_time = time.time() - start
print(f"\n3 запроса на {len(docs)} документах: {naive_time:.1f} сек")

запрос 13: найдено 16, первые docid: [76, 697, 806, 1342, 1356]


запрос 14: найдено 13, первые docid: [76, 469, 603, 699, 1420]


запрос 15: найдено 4, первые docid: [76, 286, 979, 2734]

3 запроса на 2781 документах: 1.9 сек


Секунды на три запроса по игрушечной коллекции в 20 МБ. В вебе документов не тысячи, а миллиарды.

Проблема не только в объёме: мы заново токенизируем всю коллекцию на **каждый** запрос, хотя тексты не меняются.
Значит, эту работу нужно сделать один раз заранее — в оффлайне.

## 2. Обратный индекс

Храним словарь: **термин → отсортированный список docid**, в которых он встречается. Такой список называется постинг-листом.

Сортировка списков — не формальность: на ней держится быстрое пересечение за один проход.

In [6]:
from collections import defaultdict


def build_index(docs):
    index = defaultdict(list)
    for docid in sorted(docs):
        title, body = docs[docid]
        for term in set(preprocess(title)) | set(preprocess(body)):
            index[term].append(docid)  # docid обходим по возрастанию, поэтому списки сразу отсортированы
    return dict(index)


start = time.time()
inverted_index = build_index(docs)
print(f"индекс построен за {time.time() - start:.1f} сек")
print(f"терминов в словаре: {len(inverted_index)}")
print(f"всего постингов: {sum(len(postings) for postings in inverted_index.values()):,}")
print("\nпостинг-лист слова 'война':", inverted_index["война"][:10], "...")
print("документов с этим словом:", len(inverted_index["война"]))

индекс построен за 0.8 сек
терминов в словаре: 62604
всего постингов: 808,527

постинг-лист слова 'война': [5, 28, 29, 30, 32, 49, 74, 76, 78, 80] ...
документов с этим словом: 331


In [7]:
len(inverted_index['рак'])

20

Длина постинг-листа — это document frequency, то самое df из формулы IDF. Оно пригодится в теме про текстовое ранжирование.

Теперь операции над списками. Пересечение — это И-запрос, слияние — ИЛИ-запрос.

In [8]:
def intersect(first, second):
    """И: пересечение двух отсортированных списков за один проход, O(len(first) + len(second))."""
    result, i, j = [], 0, 0
    while i < len(first) and j < len(second):
        if first[i] == second[j]:
            result.append(first[i])
            i += 1
            j += 1
        elif first[i] < second[j]:
            i += 1
        else:
            j += 1
    return result


def union(first, second):
    """ИЛИ: слияние двух отсортированных списков без дубликатов."""
    result, i, j = [], 0, 0
    while i < len(first) and j < len(second):
        if first[i] == second[j]:
            result.append(first[i])
            i += 1
            j += 1
        elif first[i] < second[j]:
            result.append(first[i])
            i += 1
        else:
            result.append(second[j])
            j += 1
    result.extend(first[i:])
    result.extend(second[j:])
    return result


print(intersect([1, 3, 5, 7], [3, 4, 5, 9]))
print(union([1, 3, 5, 7], [3, 4, 5, 9]))

[3, 5]
[1, 3, 4, 5, 7, 9]


In [9]:
def search(query_text, index):
    """Внутри группы -- ИЛИ (сливаем), между группами -- И (пересекаем)."""
    result = None
    for group in parse_query(query_text):
        group_postings = []
        for variant in group:
            group_postings = union(group_postings, index.get(variant, []))
        result = group_postings if result is None else intersect(result, group_postings)
        if not result:
            return []
    return result or []


# сверяем с наивным поиском: результат обязан быть таким же
assert search(queries[13], inverted_index) == naive_search(queries[13])
print("выдача совпала с полным перебором")

start = time.time()
found = sum(len(search(queries[qid], inverted_index)) for qid in list(queries)[:100])
index_time = time.time() - start
print(f"\n100 запросов по индексу: {index_time:.2f} сек (найдено {found} документов суммарно)")
print(f"наивный поиск на тех же данных тратил ~{naive_time / 3:.1f} сек на один запрос")

выдача совпала с полным перебором

100 запросов по индексу: 0.01 сек (найдено 1591 документов суммарно)
наивный поиск на тех же данных тратил ~0.6 сек на один запрос


Тяжёлая работа уехала в оффлайн: индекс строится один раз, а запрос обрабатывается за миллисекунды.

Посмотрим, сколько индекс занимает места.

In [10]:
import pickle

with open("my_index.pkl", "wb") as f:
    pickle.dump(inverted_index, f)

print(f"индекс на диске: {os.path.getsize('my_index.pkl') / 2**20:.1f} МБ")
print(f"средняя длина постинг-листа: {sum(map(len, inverted_index.values())) / len(inverted_index):.1f}")
print("самые частые термины:",
      [term for term, _ in sorted(inverted_index.items(), key=lambda kv: -len(kv[1]))[:10]])

индекс на диске: 3.7 МБ
средняя длина постинг-листа: 12.9
самые частые термины: ['и', 'в', 'на', 'не', 'с', 'быть', 'что', 'а', 'как', 'по']


Куда двигаться, если памяти не хватает:

- хранить не сами docid, а разности соседних (**дельты**): числа становятся маленькими;
- кодировать дельты **variable byte** или гамма-кодом вместо питоновских int по 28 байт;
- держать постинг-листы в `array('i')` или `bytes` вместо `list`;
- не загружать весь индекс в память: словарь терминов с **смещениями** в файле, а постинги читать с диска по запросу.

Первые два пункта — это тема про сжатие индекса, последний — то, как устроены настоящие движки.

Пример: постинг-лист и его дельты.

```
docid:  [0, 1, 200, 20000, 12121212, 12121213, 12121214, 12121215, 12121219]
дельты: [0, 1, 199, 19800, 12101212,        1,        1,        1,        4]
```

Почти все дельты маленькие и помещаются в 1 байт — на этом и строятся variable byte и гамма-код. Большие скачки редки, для них хватит 4 байт.

## 3. Тот же поиск в движке whoosh

`whoosh` — поисковый движок на чистом Python: медленный, зато весь его код можно прочитать.
В проде так не делают (там Lucene, Elasticsearch, Sphinx), но для учёбы это идеальный подопытный.

In [11]:
from whoosh.fields import Schema, TEXT, NUMERIC

# Схема -- это описание полей документа. Главное различие:
#   "индексируется"  -> по полю можно искать (поле попадает в обратный индекс)
#   stored=True      -> значение можно достать из выдачи и построить по нему сниппет
schema = Schema(
    docid=NUMERIC(stored=True, unique=True),  # unique=True позволяет потом обновлять документ по docid
    title=TEXT(stored=True),                  # TEXT: текст анализируется и индексируется
    body=TEXT(stored=True),                   # stored=True нужен для подсветки найденных слов
)

print(schema)

<Schema: ['body', 'docid', 'title']>


Поля бывают разные: `TEXT` анализируется, `ID` и `KEYWORD` индексируются целиком, `NUMERIC` и `BOOLEAN` — для чисел и флагов,
`STORED` только хранится, искать по нему нельзя.

У `TEXT` есть полезный параметр `phrase=False`: тогда позиции слов в индекс не пишутся,
индекс становится заметно меньше, но фразовый поиск ломается. Это ровно тот компромисс, который вам предстоит выбирать в ДЗ1.

In [12]:
import shutil
import whoosh.index as whoosh_index

INDEX_DIR = "indexdir"
shutil.rmtree(INDEX_DIR, ignore_errors=True)
os.mkdir(INDEX_DIR)

ix = whoosh_index.create_in(INDEX_DIR, schema)

# limitmb -- размер буфера в памяти. Когда буфер заполняется, whoosh сбрасывает
# на диск очередной сегмент индекса. Больше буфер -> меньше сегментов -> быстрее индексация
writer = ix.writer(limitmb=256)

start = time.time()
for docid, (title, body) in docs.items():
    writer.add_document(docid=docid, title=title, body=body)
writer.commit()  # до commit ничего не видно поиску

index_size = sum(os.path.getsize(os.path.join(INDEX_DIR, f)) for f in os.listdir(INDEX_DIR))
print(f"проиндексировано {ix.doc_count()} документов за {time.time() - start:.1f} сек")
print(f"индекс на диске: {index_size / 2**20:.1f} МБ")

проиндексировано 2781 документов за 12.1 сек
индекс на диске: 32.7 МБ


Индекс whoosh толще нашего, и понятно почему: он хранит позиции слов, частоты, длины документов и сами тексты полей (`stored=True`).

Заглянем внутрь — там тот же словарь и те же постинг-листы.

In [13]:
with ix.reader() as reader:
    lexicon = [term.decode("utf-8") for term in reader.lexicon("body")]
    print(f"терминов в словаре поля body: {len(lexicon)} (у нас в своём индексе: {len(inverted_index)})")
    print("кусок словаря:", lexicon[20000:20008], "\n")

    print("df('война') в whoosh:", reader.doc_frequency("body", "война"),
          "| в нашем индексе:", len(inverted_index["война"]))

    postings = list(reader.postings("body", "война").all_ids())
    print("постинг-лист whoosh:", postings[:10])
    print("наши docid:       ", [reader.stored_fields(docnum)["docid"] for docnum in postings[:10]])

терминов в словаре поля body: 62550 (у нас в своём индексе: 62604)
кусок словаря: ['изнеженность', 'изнеживать', 'изнемогать', 'изнеможение', 'изнемочь', 'изничтожение', 'изнова', 'износ'] 

df('война') в whoosh: 331 | в нашем индексе: 331
постинг-лист whoosh: [4, 27, 28, 29, 31, 48, 73, 75, 77, 79]
наши docid:        [5, 28, 29, 30, 32, 49, 74, 76, 78, 80]


Обратите внимание: внутри whoosh документы нумерует сам, своими последовательными номерами, а наш `docid` для него —
обычное хранимое поле. Поэтому постинг-листы whoosh и наши выглядят похоже, но состоят из разных чисел.

Теперь сам поиск.

In [14]:
from whoosh.qparser import OperatorsPlugin, QueryParser


def make_parser(field="body", **kwargs):
    parser = QueryParser(field, schema=schema, **kwargs)
    # по умолчанию ИЛИ пишется словом OR, а в наших запросах это '|'
    parser.replace_plugin(OperatorsPlugin(Or=r"\|"))
    return parser


parser = make_parser()
query = parser.parse(queries[13])
print("оригинальный запрос:", queries[13], "\n")
print("разобранный запрос:", query, "\n")

with ix.searcher() as searcher:
    results = searcher.search(query, limit=5) # топ-5 документов по bm25f
    print("всего найдено:", len(results), "| показано:", results.scored_length(), "\n")
    for hit in results:
        print(f"docid={hit['docid']} score={hit.score:.1f} {hit['title'][:60]}")
        print("   ", hit.highlights("body")[:160].replace("\n", " "), "\n")

оригинальный запрос: (1|I) (ВОЙНА|БОЙ|СРАЖЕНИЕ|БИТВА) (ГЕРМАНИЯ|ФРГ|GERMANY) (СОЮЗ|СССР|ОБЪЕДИНЕНИЕ) 

разобранный запрос: ((body:война OR body:бой OR body:сражение OR body:битва) AND (body:германия OR body:фрг OR body:germany) AND (body:союз OR body:ссср OR body:объединение)) 

всего найдено: 32 | показано: 5 

docid=2159 score=33.9 ПЕРВЫЙ ПЕРИОД ВОЙНА СЕНТЯБРЬ 1939 ИЮНЬ 1941
    23 АВГУСТ МЕЖДУ <b class="match term0">ГЕРМАНИЯ</b> И <b class="match term1">СССР</b> БЫТЬ ПОДПИСАТЬ ДОГОВОР...ПРОТОКОЛ К ДОГОВОР <b class="match term1">СССР</b 

docid=2161 score=32.7 29 ИЮНЬ ГЕРМАНО ФИНСКИЙ ВОЙСКА ПРЕДПРИНИМАТЬ НАСТУПЛЕНИЕ В З
    ЮЖНО-АФРИКАНСКИЙ <b class="match term2">СОЮЗ</b> АВСТРАЛИЯ ИНДИЯ И <b class="match term1">СССР</b> С КОНЕЦ АВГУСТ 1941...БРАЗИЛИЯ ОБЪЯВЛЯТЬ <b class="match term 

docid=2160 score=31.9 СРАЖЕНИЕ В АФРИКА СРЕДИЗЕМНОМОРЬЕ И НА БАЛКАНЫ
    ТРЕБОВАНИЕ К <b class="match term1">СССР</b> ПЕРЕД НАЧАЛО <b class="match term3">ВОЙНА</b> И ДРУГОЙ В КОНЕЦ ДЕКАБРЬ...И ЧТО СЛУХ О

Что здесь произошло:

- пробел между словами парсер понял как И, `|` — как ИЛИ, и построил дерево запроса;
- `limit=5` ограничивает выдачу, но `len(results)` всё равно показывает полное число найденного;
- документы отранжированы: у каждого есть `score`, по умолчанию BM25F — это тема лекции про текстовое ранжирование;
- `highlights` построил сниппет по полю `body`, для этого оно и хранится.

Сравним выдачу движка с нашей.

In [15]:
with ix.searcher() as searcher:
    for qid in (13, 14, 15):
        whoosh_ids = {hit["docid"] for hit in searcher.search(make_parser().parse(queries[qid]), limit=None)}
        our_ids = set(search(queries[qid], inverted_index))
        print(f"запрос {qid}: whoosh {len(whoosh_ids)}, наш индекс {len(our_ids)}, "
              f"общих {len(whoosh_ids & our_ids)}, только у нас {len(our_ids - whoosh_ids)}")

запрос 13: whoosh 32, наш индекс 16, общих 16, только у нас 0
запрос 14: whoosh 50, наш индекс 13, общих 13, только у нас 0
запрос 15: whoosh 8, наш индекс 4, общих 4, только у нас 0


Whoosh находит заметно больше. Причина не в индексе, а в **анализаторе**: перед индексацией whoosh сам режет текст на токены и часть выбрасывает.

(Наш индекс построен по `title` и `body` сразу, а whoosh ищет только по `body`. В этой коллекции title — это начало body, так что на числа это не влияет.)

In [16]:
from whoosh.analysis import StandardAnalyzer

sample = "(1|I) ВОЙНА THE ГЕРМАНИЯ x2"
print("whoosh:", [token.text for token in StandardAnalyzer()(sample)])
print("наш preprocess:", preprocess(sample))

whoosh: ['война', 'германия', 'x2']
наш preprocess: ['1', 'i', 'война', 'the', 'германия', 'x2']


`StandardAnalyzer` по умолчанию выбрасывает токены короче двух символов и английские стоп-слова.
Поэтому `1`, `I` и `the` из запроса исчезают, а русские стоп-слова остаются: список стоп-слов у whoosh английский.

Вот и ответ, почему выдачи разошлись: у нас группа `(1|I)` требует, чтобы в документе было «1» или «i»,
а whoosh выбросил её целиком и ищет на одно условие меньше.

**Это прямо относится к ДЗ1.** Правильный ответ там считается токенизацией из шаблона, поэтому
`preprocess()` нужно брать из шаблона и применять одинаково и к документам, и к запросам. Любая своя «улучшенная»
токенизация разведёт вашу выдачу с эталоном.

Последний сюжет: строгое И на длинных запросах почти ничего не находит.

In [17]:
from whoosh import qparser

soft_parser = make_parser(group=qparser.OrGroup.factory(0.9))  # мягкое ИЛИ: чем больше слов совпало, тем выше документ

with ix.searcher() as searcher:
    for qid in (11, 12, 13):
        strict = len(searcher.search(make_parser().parse(queries[qid]), limit=None))
        soft = len(searcher.search(soft_parser.parse(queries[qid]), limit=None))
        print(f"запрос {qid}: строгое И -> {strict:>4} документов | мягкое ИЛИ -> {soft:>4}")
        print("   ", queries[qid][:100], "\n")

запрос 11: строгое И ->    0 документов | мягкое ИЛИ -> 1969
    (1|I) 89161234567 (НАЙТИ|ПОИСК|ИСКАТЬ|ВЫЯСНИТЬ) КОМУ (ПРИНАДЛЕЖАТЬ|ИМЕТЬ) (ЭТОТ|ETOT) (НОМЕР|КОМНАТА 

запрос 12: строгое И ->    0 документов | мягкое ИЛИ ->  114
    (1|I) (ТЕЛЕФОН|МОБИЛЬНИК|СМАРТФОН) ОТЗИВ 

запрос 13: строгое И ->   32 документов | мягкое ИЛИ ->  690
    (1|I) (ВОЙНА|БОЙ|СРАЖЕНИЕ|БИТВА) (ГЕРМАНИЯ|ФРГ|GERMANY) (СОЮЗ|СССР|ОБЪЕДИНЕНИЕ) 



Первый запрос — длинный, с номером телефона: по строгому И не находится ничего.
Второй — с опечаткой «ОТЗИВ»: тоже ноль, и никакое смягчение И тут не поможет, нужен исправитель опечаток.

Отсюда растут две следующие темы курса: отбор кандидатов мягким И вместо строгого и исправление запросов.


## 4. Задания

**Задание 1. Ужать индекс.**
Сейчас постинг-листы — это питоновские списки, и каждый docid в них занимает под 36 байт: 8 байт указателя и 28 байт объекта int.
Переведите индекс на хранение **разностей соседних docid** в `array("i")` (4 байта на число), посчитайте, сколько памяти
занимали списки и сколько занимают массивы, и проверьте, что выдача `search` не изменилась.

**Задание 2. Найти расхождения с whoosh.**
Прогоните 50 запросов через свой индекс и через whoosh и соберите документы, которые нашлись только в одном из них.
Посмотрите на такие запросы и объясните расхождение.

In [18]:
# Задание 1: ваш код


In [19]:
# Задание 2: ваш код


## 5. Что дальше: ДЗ1

В домашнем задании нужно сделать то же, что мы делали во второй части, но аккуратно и под ограничениями.
Готовые поисковые движки использовать нельзя — индекс пишем сами.

Требования (подробности в [README задания](../../homeworks/hw1-boolean-retrieval/)):

- `solution.py` работает в двух режимах: построение индекса (`--build_index --index_dir=...`) и генерация сабмишна;
- токенизация только через `preprocess()` из шаблона;
- многословный запрос трактуется как И-запрос;
- в режиме генерации сабмишна решение должно укладываться в 150 МБ памяти;
- скор проверяется на Kaggle, код сдаётся на платформе курса.

Полезно помнить из сегодняшнего: сортированные постинг-листы, пересечение за один проход и компактное представление docid.
Ещё одна привычка из той же серии: когда групп в запросе много, пересекать их стоит начиная с самых коротких постинг-листов —
тогда промежуточный результат маленький с самого начала.